# Train Catch with Disco103 in 10 lines

**[Disco103](https://doi.org/10.1038/s41586-025-09761-x)** is a meta-learned RL update rule from DeepMind (Nature, 2025). Instead of hand-crafted loss functions like PPO, a small neural network generates loss targets for your agent.

This notebook trains a simple agent to play Catch (8Ã—8 grid, catch a falling ball) using `disco-torch` â€” the PyTorch port. It reaches **99% catch rate** in ~1000 steps.

**Runtime**: ~2 hours on T4, ~1.5 hours on A100.

In [ ]:
#@title 1. Install
!pip install -q "disco-torch[hub]>=0.4.0"

In [ ]:
#@title 2. Environment + Agent
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class CatchEnv:
    """Catch: ball falls down 8x8 grid, paddle catches at bottom."""

    def __init__(self, num_envs=2, rows=8, cols=8):
        self.num_envs, self.rows, self.cols = num_envs, rows, cols
        self.ball_row = np.zeros(num_envs, dtype=np.int32)
        self.ball_col = np.zeros(num_envs, dtype=np.int32)
        self.paddle_col = np.zeros(num_envs, dtype=np.int32)
        self.reset()

    def reset(self, mask=None):
        if mask is None:
            mask = np.ones(self.num_envs, dtype=bool)
        self.ball_row[mask] = 0
        self.ball_col[mask] = np.random.randint(0, self.cols, size=mask.sum())
        self.paddle_col[mask] = self.cols // 2

    def step(self, actions):
        self.paddle_col = np.clip(
            self.paddle_col + actions.astype(np.int32) - 1, 0, self.cols - 1
        )
        self.ball_row += 1
        done = self.ball_row >= self.rows - 1
        rewards = np.zeros(self.num_envs, dtype=np.float32)
        rewards[done & (self.ball_col == self.paddle_col)] = 1.0
        rewards[done & (self.ball_col != self.paddle_col)] = -1.0
        self.reset(mask=done)
        return rewards, done.astype(np.float32)

    def obs(self):
        grid = np.zeros((self.num_envs, self.rows, self.cols), dtype=np.float32)
        for i in range(self.num_envs):
            grid[i, self.ball_row[i], self.ball_col[i]] = 1.0
            grid[i, self.rows - 1, self.paddle_col[i]] = 1.0
        return grid


class DiscoMLPAgent(nn.Module):
    """Reference agent: feedforward MLP + action-conditional LSTM."""

    def __init__(self, obs_dim=64, num_actions=3, prediction_size=600, num_bins=601,
                 head_init_std=1e-2):
        super().__init__()
        self.num_actions = num_actions
        A = num_actions
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU()
        )
        self.policy_head = nn.Linear(512, A)
        self.y_head = nn.Linear(512, prediction_size)
        self.cell_init = nn.Linear(512, 128)
        self.model_lstm = nn.LSTMCell(A, 128)
        self.z_mlp = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, prediction_size))
        self.q_mlp = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, num_bins))
        self.aux_pi_mlp = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, A))

        # Small init for output heads â€” critical for stable Disco103 training
        nn.init.normal_(self.policy_head.weight, std=head_init_std)
        nn.init.zeros_(self.policy_head.bias)
        nn.init.normal_(self.y_head.weight, std=head_init_std)
        nn.init.zeros_(self.y_head.bias)
        for head in [self.z_mlp, self.q_mlp, self.aux_pi_mlp]:
            nn.init.normal_(head[-1].weight, std=head_init_std)
            nn.init.zeros_(head[-1].bias)

    def init_lstm_state(self, B, device=None):
        return (torch.zeros(B, 128, device=device), torch.zeros(B, 128, device=device))

    def forward_step(self, obs, lstm_state, should_reset=None):
        B, A = obs.shape[0], self.num_actions
        emb = self.backbone(obs)
        logits = self.policy_head(emb)
        y = self.y_head(emb)
        cell = self.cell_init(emb)
        h, c = torch.tanh(cell), cell
        z_list, q_list, aux_list = [], [], []
        for a in range(A):
            one_hot = torch.zeros(B, A, device=obs.device)
            one_hot[:, a] = 1.0
            h, c = self.model_lstm(one_hot, (h, c))
            z_list.append(self.z_mlp(h))
            q_list.append(self.q_mlp(h))
            aux_list.append(self.aux_pi_mlp(h))
        return {
            "logits": logits, "y": y,
            "z": torch.stack(z_list, 1),
            "q": torch.stack(q_list, 1),
            "aux_pi": torch.stack(aux_list, 1),
        }, lstm_state

    def forward(self, obs_seq, should_reset=None):
        T, B = obs_seq.shape[:2]
        dummy = self.init_lstm_state(B, obs_seq.device)
        outs = [self.forward_step(obs_seq[t], dummy)[0] for t in range(T)]
        return {k: torch.stack([o[k] for o in outs]) for k in outs[0]}


print(f"Agent params: {sum(p.numel() for p in DiscoMLPAgent().parameters()):,}")
print("Setup complete.")

In [ ]:
#@title 3. Train
import time
from collections import deque
from disco_torch import DiscoTrainer, collect_rollout

# Setup
agent = DiscoMLPAgent().to(device)
trainer = DiscoTrainer(agent, device=device, lr=0.01)

env = CatchEnv(num_envs=2)
obs = env.obs()
lstm_state = agent.init_lstm_state(env.num_envs, device)

def step_fn(actions):
    rewards, dones = env.step(actions)
    return env.obs(), rewards, dones

# Train
NUM_STEPS = 1000
completed = deque(maxlen=100)
all_returns = []
log_data = []
t0 = time.time()

for step in range(NUM_STEPS):
    rollout, obs, lstm_state = collect_rollout(
        agent, step_fn, obs, lstm_state, rollout_len=29, device=device,
    )

    # Track episodes
    rewards_np = rollout["rewards"].cpu().numpy()
    discounts_np = rollout["discounts"].cpu().numpy()
    for t in range(rewards_np.shape[0]):
        for b in range(rewards_np.shape[1]):
            if discounts_np[t, b] == 0.0:
                completed.append(float(rewards_np[t, b]))
                all_returns.append(float(rewards_np[t, b]))

    logs = trainer.step(rollout)

    if (step + 1) % 50 == 0:
        catch = sum(1 for r in completed if r > 0) / len(completed) if completed else 0
        elapsed = time.time() - t0
        eta = elapsed / (step + 1) * NUM_STEPS - elapsed
        log_data.append((step + 1, catch))
        print(
            f"Step {step+1:5d} | catch={catch:.0%} | "
            f"loss={logs.get('total_loss', 0):.3f} | "
            f"grads={logs['grad_steps']} | {elapsed:.0f}s (eta {eta:.0f}s)"
        )

final = sum(1 for r in all_returns[-100:] if r > 0) / min(len(all_returns), 100)
print(f"\nFinal catch rate (last 100): {final:.0%}")
if final >= 0.8:
    print("PASS: Disco103 learns Catch.")

# Plot
import matplotlib.pyplot as plt

rates = []
for i in range(len(all_returns)):
    chunk = all_returns[max(0, i - 99): i + 1]
    rates.append(sum(1 for r in chunk if r > 0) / len(chunk))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

if log_data:
    steps, catches = zip(*log_data)
    ax1.plot(steps, catches, "b-o", markersize=3)
ax1.axhline(y=0.8, color="g", linestyle="--", alpha=0.5, label="80% target")
ax1.axhline(y=0.125, color="gray", linestyle=":", alpha=0.5, label="Random")
ax1.set(xlabel="Training Step", ylabel="Catch Rate", title="Learning Curve")
ax1.set_ylim(-0.05, 1.05)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(rates, "b-", alpha=0.7)
ax2.axhline(y=0.8, color="g", linestyle="--", alpha=0.5)
ax2.set(xlabel="Episode", ylabel="Catch Rate (rolling 100)", title="Per-Episode")
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()